# Pandas Practice 2
## Data Processing - ETL Pipeline
Reading a large CSV file from a URL, identifying stringified JSON columns, parsing and flattening nested data, cleaning and writing to a new CSV file.

In [47]:
import pandas as pd
import ast

In [48]:
url = "https://media.githubusercontent.com/media/BellexFire/Movies-ETL/refs/heads/main/movies_metadata.csv"
# Load the dataset from the provided URL
df = pd.read_csv(url, low_memory=False) #, low_memory=False to avoid dtype warning
df.shape

(45466, 24)

In [49]:
df.columns.tolist()

['adult',
 'belongs_to_collection',
 'budget',
 'genres',
 'homepage',
 'id',
 'imdb_id',
 'original_language',
 'original_title',
 'overview',
 'popularity',
 'poster_path',
 'production_companies',
 'production_countries',
 'release_date',
 'revenue',
 'runtime',
 'spoken_languages',
 'status',
 'tagline',
 'title',
 'video',
 'vote_average',
 'vote_count']

In [50]:
df.dtypes


adult                        str
belongs_to_collection        str
budget                       str
genres                       str
homepage                     str
id                           str
imdb_id                      str
original_language            str
original_title               str
overview                     str
popularity                   str
poster_path                  str
production_companies         str
production_countries         str
release_date                 str
revenue                  float64
runtime                  float64
spoken_languages             str
status                       str
tagline                      str
title                        str
video                     object
vote_average             float64
vote_count               float64
dtype: object

In [51]:
df_clean = df.copy()

In [52]:
df['popularity'] 

0        21.946943
1        17.015539
2          11.7129
3         3.859495
4         8.387519
           ...    
45461     0.072051
45462     0.178241
45463     0.903007
45464     0.003503
45465     0.163015
Name: popularity, Length: 45466, dtype: str

In [53]:
numreric_cols = ['budget', 'popularity', 'revenue', 'runtime', 'vote_average', 'vote_count']
for col in numreric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
df_clean['release_date'] = pd.to_datetime(df_clean['release_date'], errors='coerce')
df_clean.dtypes

adult                               str
belongs_to_collection               str
budget                          float64
genres                              str
homepage                            str
id                                  str
imdb_id                             str
original_language                   str
original_title                      str
overview                            str
popularity                      float64
poster_path                         str
production_companies                str
production_countries                str
release_date             datetime64[us]
revenue                         float64
runtime                         float64
spoken_languages                    str
status                              str
tagline                             str
title                               str
video                            object
vote_average                    float64
vote_count                      float64
dtype: object

### Parsing Stringified JSON Columns

In [54]:
df_clean['genres'] = df['genres'].copy()

df_clean['genres'] = df_clean['genres'].apply(
    lambda x: ' | '.join([genre['name'] for genre in ast.literal_eval(x)])
    if pd.notnull(x) and isinstance(x, str)  else ''
)



In [55]:
df_clean[['title', 'genres']].head()

,title,genres
0,Toy Story,Animation | Comedy | Family
1,Jumanji,Adventure | Fantasy | Family
2,Grumpier Old Men,Romance | Comedy
3,Waiting to Exhale,Comedy | Drama | Romance
4,Father of the Bride Part II,Comedy


In [56]:
df_clean['production_companies']= df_clean['production_companies'].apply(
    lambda x : '|'.join([company['name'] for company in ast.literal_eval(x)]) if pd.notnull(x) and isinstance(x, str) and x.strip().startswith('[') else ''
)
df_clean[['title', 'production_companies']].head()

,title,production_companies
0,Toy Story,Pixar Animation Studios
1,Jumanji,TriStar Pictures|Teitler Film|Interscope Commu...
2,Grumpier Old Men,Warner Bros.|Lancaster Gate
3,Waiting to Exhale,Twentieth Century Fox Film Corporation
4,Father of the Bride Part II,Sandollar Productions|Touchstone Pictures


In [57]:
df_clean['production_countries'] = df_clean['production_countries'].apply(
    lambda x : '|'.join([country['name'] for country in ast.literal_eval(x)]) if pd.notnull(x) and isinstance(x, str) and x.strip().startswith('[') else ''
)
df_clean[['title', 'production_countries']].head()

,title,production_countries
0,Toy Story,United States of America
1,Jumanji,United States of America
2,Grumpier Old Men,United States of America
3,Waiting to Exhale,United States of America
4,Father of the Bride Part II,United States of America


In [58]:
df_clean['spoken_languages'] = df_clean['spoken_languages'].apply(
    lambda x : '|'.join([language['name'] for language in ast.literal_eval(x)]) if pd.notnull(x) and isinstance(x, str) and x.strip().startswith('[') else ''
)
df_clean.loc[:, ['title', 'spoken_languages']].head()

,title,spoken_languages
0,Toy Story,English
1,Jumanji,English|Français
2,Grumpier Old Men,English
3,Waiting to Exhale,English
4,Father of the Bride Part II,English


In [59]:
df_clean['collection_name'] = df_clean['belongs_to_collection'].apply(
    lambda x : ast.literal_eval(x)['name'] if pd.notnull(x) and isinstance(x, str) and x.strip().startswith('{') else ''
)
df_clean.loc[:, ['title', 'collection_name']].head()

,title,collection_name
0,Toy Story,Toy Story Collection
1,Jumanji,
2,Grumpier Old Men,Grumpy Old Men Collection
3,Waiting to Exhale,
4,Father of the Bride Part II,Father of the Bride Collection


In [60]:
df_clean['collection_poster_path'] = df_clean['belongs_to_collection'].apply(
    lambda x : ast.literal_eval(x)['poster_path'] if pd.notnull(x) and isinstance(x, str) and x.strip().startswith('{') else ''
)
df_clean.loc[:, ['title', 'collection_poster_path']].head()

,title,collection_poster_path
0,Toy Story,/7G9915LfUQ2lVfwMEEhDsn3kT4B.jpg
1,Jumanji,
2,Grumpier Old Men,/nLvUdqgPgm3F85NMCii9gVFUcet.jpg
3,Waiting to Exhale,
4,Father of the Bride Part II,/nts4iOmNnq7GNicycMJ9pSAn204.jpg


In [61]:
df_clean['collection_backdrop_path'] = df_clean['belongs_to_collection'].apply(
    lambda x : ast.literal_eval(x)['backdrop_path'] if pd.notnull(x) and isinstance(x, str) and x.strip().startswith('{') else ''
)
df_clean.loc[:, ['title', 'collection_backdrop_path']].head()

,title,collection_backdrop_path
0,Toy Story,/9FBwqcd9IRruEDUrTdcaafOMKUq.jpg
1,Jumanji,
2,Grumpier Old Men,/hypTnLot2z8wpFS7qwsQHW1uV8u.jpg
3,Waiting to Exhale,
4,Father of the Bride Part II,/7qwE57OVZmMJChBpLEbJEmzUydk.jpg


In [65]:
df_clean.drop(columns=['adult', 'belongs_to_collection'], inplace=True)

KeyError: "['adult', 'belongs_to_collection'] not found in axis"

In [63]:
df_clean.head(5)

,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,popularity,poster_path,...,spoken_languages,status,tagline,title,video,vote_average,vote_count,collection_name,collection_poster_path,collection_backdrop_path
0,30000000.0,Animation | Comedy | Family,http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",21.946943,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,...,English,Released,NaN,Toy Story,False,7.7,5415.0,Toy Story Collection,/7G9915LfUQ2lVfwMEEhDsn3kT4B.jpg,/9FBwqcd9IRruEDUrTdcaafOMKUq.jpg
1,65000000.0,Adventure | Fantasy | Family,NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,17.015539,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,...,English|Français,Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0,,,
2,0.0,Romance | Comedy,NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,11.712900,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,...,English,Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0,Grumpy Old Men Collection,/nLvUdqgPgm3F85NMCii9gVFUcet.jpg,/hypTnLot2z8wpFS7qwsQHW1uV8u.jpg
3,16000000.0,Comedy | Drama | Romance,NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",3.859495,/16XOMpEaLWkrcPqSQqhTmeJuqQl.jpg,...,English,Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0,,,
4,0.0,Comedy,NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,8.387519,/e64sOI48hQXyru7naBFyssKFxVd.jpg,...,English,Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0,Father of the Bride Collection,/nts4iOmNnq7GNicycMJ9pSAn204.jpg,/7qwE57OVZmMJChBpLEbJEmzUydk.jpg


In [64]:
df_clean.shape

(45466, 25)

In [66]:
df_clean.columns.tolist()

['budget',
 'genres',
 'homepage',
 'id',
 'imdb_id',
 'original_language',
 'original_title',
 'overview',
 'popularity',
 'poster_path',
 'production_companies',
 'production_countries',
 'release_date',
 'revenue',
 'runtime',
 'spoken_languages',
 'status',
 'tagline',
 'title',
 'video',
 'vote_average',
 'vote_count',
 'collection_name',
 'collection_poster_path',
 'collection_backdrop_path']

In [67]:
df_clean.to_csv('movies_cleaned.csv', index=False)
print("Saved!")

Saved!
